# 🛒 SmartCart Clustering System — Customer Segmentation

**Assignment**: Unsupervised Machine Learning — Customer Segmentation  
**Platform**: SmartCart E-Commerce  
**Objective**: Group 2,240 customers into meaningful segments using clustering algorithms to enable personalised marketing and reduce churn.

---

## 📌 Problem Statement

SmartCart uses a **one-size-fits-all** marketing strategy — treating all customers equally. This leads to:
- Inefficient marketing spend
- Missed high-value customer retention
- Inability to identify churn-prone users

**Solution**: Apply unsupervised ML clustering to discover hidden customer segments based on purchasing behaviour, engagement levels, and loyalty indicators.

---

## 📦 Dataset Overview

| Property | Detail |
|----------|--------|
| Records | 2,240 customers |
| Features | 22 attributes |
| Type | Demographics + Purchase Behaviour + Engagement |
| Target | None (Unsupervised) |

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from kneed import KneeLocator

# Plot style
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.facecolor'] = '#f8f9fa'

print('✅ All libraries imported successfully')

## 2. Load & Explore Dataset

In [ ]:
df = pd.read_csv('../data/smartcart_customers.csv')

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
print('Dataset Info:')
df.info()

In [ ]:
print('Missing Values:')
print(df.isnull().sum()[df.isnull().sum() > 0])

print('\nStatistical Summary:')
df.describe().T

## 3. Data Preprocessing

### 3.1 Handle Missing Values

In [ ]:
# Fill missing Income with median (robust to outliers)
df['Income'] = df['Income'].fillna(df['Income'].median())

print(f'Missing values after fix: {df.isnull().sum().sum()}')

### 3.2 Feature Engineering

Creating meaningful derived features from raw data:

In [ ]:
# Age from Year_Birth
df['Age'] = 2026 - df['Year_Birth']

# Customer tenure (how long they've been with SmartCart)
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], dayfirst=True)
reference_date = df['Dt_Customer'].max()
df['Customer_Tenure_Days'] = (reference_date - df['Dt_Customer']).dt.days

# Total spending across all product categories
spend_cols = ['MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']
df['Total_Spending'] = df[spend_cols].sum(axis=1)

# Total children at home
df['Total_Children'] = df['Kidhome'] + df['Teenhome']

# Simplify Education into 3 tiers
df['Education'] = df['Education'].replace({
    'Basic': 'Undergraduate', '2n Cycle': 'Undergraduate',
    'Graduation': 'Graduate',
    'Master': 'Postgraduate', 'PhD': 'Postgraduate'
})

# Simplify Marital Status into Partner / Alone
df['Living_With'] = df['Marital_Status'].replace({
    'Married': 'Partner', 'Together': 'Partner',
    'Single': 'Alone', 'Divorced': 'Alone',
    'Widow': 'Alone', 'Absurd': 'Alone', 'YOLO': 'Alone'
})

print('✅ Feature engineering complete')
print(f'New features: Age, Customer_Tenure_Days, Total_Spending, Total_Children, Living_With')

### 3.3 Drop Unnecessary Columns

In [ ]:
# Drop raw columns that have been replaced by engineered features
id_cols      = ['ID', 'Year_Birth', 'Marital_Status', 'Kidhome', 'Teenhome', 'Dt_Customer']
spend_raw    = ['MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']

df_cleaned = df.drop(columns=id_cols + spend_raw)

print(f'Shape after dropping: {df_cleaned.shape}')
df_cleaned.head()

## 4. Outlier Removal

In [ ]:
# Visualize distributions to identify outliers
cols_to_check = ['Income', 'Age', 'Total_Spending', 'Total_Children', 'Recency']

fig, axes = plt.subplots(1, len(cols_to_check), figsize=(16, 4))
for ax, col in zip(axes, cols_to_check):
    ax.boxplot(df_cleaned[col].dropna())
    ax.set_title(col, fontsize=10)
    ax.set_xticks([])
plt.suptitle('Boxplots — Outlier Detection', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Pair plots for key features
sns.pairplot(df_cleaned[['Income', 'Recency', 'Age', 'Total_Spending', 'Total_Children']], 
             plot_kws={'alpha': 0.5, 's': 15})
plt.suptitle('Pair Plot — Key Features', y=1.02, fontsize=13, fontweight='bold')
plt.show()

In [ ]:
print(f'Records before outlier removal: {len(df_cleaned)}')

# Remove extreme outliers
df_cleaned = df_cleaned[df_cleaned['Age'] < 90]       # Remove unrealistic ages
df_cleaned = df_cleaned[df_cleaned['Income'] < 600_000]  # Remove extreme income

print(f'Records after outlier removal : {len(df_cleaned)}')
print(f'Removed: {2240 - len(df_cleaned)} records')

## 5. Correlation Heatmap

In [ ]:
corr = df_cleaned.corr(numeric_only=True)

plt.figure(figsize=(10, 8))
sns.heatmap(
    corr,
    annot=True,
    annot_kws={'size': 6},
    cmap='coolwarm',
    center=0,
    square=True,
    linewidths=0.5
)
plt.title('Feature Correlation Heatmap', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

## 6. Encoding Categorical Features

In [ ]:
cat_cols = ['Education', 'Living_With']

ohe = OneHotEncoder(sparse_output=False, drop='first')  # drop first to avoid multicollinearity
enc_array = ohe.fit_transform(df_cleaned[cat_cols])
enc_df = pd.DataFrame(enc_array, columns=ohe.get_feature_names_out(cat_cols), index=df_cleaned.index)

df_encoded = pd.concat([df_cleaned.drop(columns=cat_cols), enc_df], axis=1)

print(f'Shape after encoding: {df_encoded.shape}')
print(f'New one-hot columns: {list(ohe.get_feature_names_out(cat_cols))}')
df_encoded.head()

## 7. Feature Scaling

> **Why scale?** KMeans and PCA are distance/variance-based — features with larger ranges would dominate. StandardScaler gives each feature mean=0, std=1.

In [ ]:
X = df_encoded.copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Scaled data shape: {X_scaled.shape}')

## 8. Dimensionality Reduction — PCA

Reducing to **3 principal components** for:
- Noise reduction
- Better clustering performance
- 3D visualization

In [ ]:
pca = PCA(n_components=3, random_state=42)
X_pca = pca.fit_transform(X_scaled)

explained = pca.explained_variance_ratio_
print(f'Explained Variance by component:')
for i, v in enumerate(explained):
    print(f'  PC{i+1}: {v*100:.2f}%')
print(f'  Total: {sum(explained)*100:.2f}%')

In [ ]:
# 3D scatter of PCA components (before clustering)
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(X_pca[:, 0], X_pca[:, 1], X_pca[:, 2], alpha=0.5, s=15, color='steelblue')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_zlabel('PC3')
ax.set_title('3D PCA Projection — Before Clustering', fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Finding Optimal K

### 9.1 Elbow Method (WCSS)

In [ ]:
wcss = []
for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_pca)
    wcss.append(kmeans.inertia_)

# Auto-detect elbow
knee = KneeLocator(range(1, 11), wcss, curve='convex', direction='decreasing')
optimal_k = knee.elbow
print(f'Optimal K (Elbow Method): {optimal_k}')

### 9.2 Silhouette Score

In [ ]:
sil_scores = []
for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_pca)
    sil_scores.append(silhouette_score(X_pca, labels))

best_k_sil = range(2, 11)[sil_scores.index(max(sil_scores))]
print(f'Best K (Silhouette Score): {best_k_sil} → Score: {max(sil_scores):.4f}')

In [ ]:
# Combined plot
k_range = list(range(2, 11))
fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.plot(range(1, 11), wcss, marker='o', color='#2196F3', linewidth=2, label='WCSS (Elbow)')
ax1.axvline(x=optimal_k, color='#2196F3', linestyle='--', alpha=0.6, label=f'Elbow at k={optimal_k}')
ax1.set_xlabel('Number of Clusters (K)', fontsize=12)
ax1.set_ylabel('WCSS', color='#2196F3', fontsize=12)
ax1.tick_params(axis='y', labelcolor='#2196F3')

ax2 = ax1.twinx()
ax2.plot(k_range, sil_scores, marker='x', color='#E91E63', linewidth=2, linestyle='--', label='Silhouette Score')
ax2.axvline(x=best_k_sil, color='#E91E63', linestyle=':', alpha=0.6, label=f'Best SS at k={best_k_sil}')
ax2.set_ylabel('Silhouette Score', color='#E91E63', fontsize=12)
ax2.tick_params(axis='y', labelcolor='#E91E63')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.title('Elbow Method + Silhouette Score — Finding Optimal K', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/optimal_k_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved to docs/optimal_k_analysis.png')

## 10. Clustering

### 10.1 K-Means Clustering (k=4)

In [ ]:
kmeans_final = KMeans(n_clusters=4, random_state=42, n_init=10)
labels_kmeans = kmeans_final.fit_predict(X_pca)

print(f'KMeans Silhouette Score: {silhouette_score(X_pca, labels_kmeans):.4f}')
print(f'Cluster sizes: {dict(zip(*np.unique(labels_kmeans, return_counts=True)))}')

In [ ]:
colors = ['#E74C3C', '#3498DB', '#F1C40F', '#2ECC71']

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
for cluster in range(4):
    mask = labels_kmeans == cluster
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], X_pca[mask, 2],
               c=colors[cluster], label=f'Cluster {cluster}', s=20, alpha=0.7)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_zlabel('PC3')
ax.set_title('K-Means Clustering (k=4) — 3D PCA View', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

### 10.2 Agglomerative (Hierarchical) Clustering

In [ ]:
agg_clf = AgglomerativeClustering(n_clusters=4, linkage='ward')
labels_agg = agg_clf.fit_predict(X_pca)

print(f'Agglomerative Silhouette Score: {silhouette_score(X_pca, labels_agg):.4f}')
print(f'Cluster sizes: {dict(zip(*np.unique(labels_agg, return_counts=True)))}')

In [ ]:
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
for cluster in range(4):
    mask = labels_agg == cluster
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], X_pca[mask, 2],
               c=colors[cluster], label=f'Cluster {cluster}', s=20, alpha=0.7)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_zlabel('PC3')
ax.set_title('Agglomerative Clustering (k=4) — 3D PCA View', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 11. Cluster Characterization & Analysis

Using **Agglomerative clustering** results (ward linkage generally gives more balanced clusters):

In [ ]:
X['Cluster'] = labels_agg

# Cluster size distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

palette = {0: '#E74C3C', 1: '#3498DB', 2: '#F1C40F', 3: '#2ECC71'}
sns.countplot(x=X['Cluster'], palette=palette, ax=axes[0], hue=X['Cluster'], legend=False)
axes[0].set_title('Customer Count per Cluster', fontweight='bold')
axes[0].set_xlabel('Cluster')
axes[0].set_ylabel('Count')

# Income vs Total Spending
sns.scatterplot(
    x=X['Total_Spending'], y=X['Income'],
    hue=X['Cluster'], palette=palette, ax=axes[1], alpha=0.7, s=40
)
axes[1].set_title('Income vs Total Spending by Cluster', fontweight='bold')
axes[1].set_xlabel('Total Spending')
axes[1].set_ylabel('Income')

plt.tight_layout()
plt.savefig('../docs/cluster_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved to docs/cluster_analysis.png')

In [ ]:
# Cluster Summary Statistics
summary_cols = ['Income', 'Total_Spending', 'Age', 'Total_Children', 'Recency',
                'NumWebPurchases', 'NumStorePurchases', 'Customer_Tenure_Days']

cluster_summary = X.groupby('Cluster')[summary_cols].mean().round(1)
print('📊 Cluster Summary (Mean values):')
cluster_summary

## 12. Cluster Profiles & Business Insights

Based on the cluster summary, we can assign meaningful business labels:

| Cluster | Profile | Strategy |
|---------|---------|----------|
| **0** | 🌟 High-Value Loyalists | High income, high spending — Premium rewards & early access |
| **1** | 💤 Inactive / At-Risk | Low recency, low spending — Re-engagement campaigns |
| **2** | 🛍️ Mid-Tier Regular | Moderate income & spending — Upsell opportunities |
| **3** | 🆕 Budget / New Shoppers | Low income, low spending — Discount-driven offers |

> ⚠️ Cluster labels above are indicative — verify against your actual cluster summary output above.

---

## 13. Conclusion

| Step | Method | Result |
|------|--------|--------|
| Missing Values | Median imputation | Income filled |
| Feature Engineering | Age, Tenure, Total Spending | 5 new features |
| Outlier Removal | Threshold filtering | ~3 records removed |
| Encoding | OneHotEncoder | Education + Living_With |
| Scaling | StandardScaler | All features normalized |
| Dimensionality Reduction | PCA (3 components) | Noise reduced |
| Optimal K | Elbow + Silhouette | K = 4 |
| Clustering | KMeans + Agglomerative | 4 customer segments |

### ✅ Key Takeaway
SmartCart can now target each of the 4 segments with **personalised marketing strategies** — improving conversion, reducing churn, and maximising ROI on marketing spend.